# MEG-Mamba feature extraction, run on Colab

`mamba_ssm` needs CUDA, so this one cell of the pipeline runs on a Colab GPU runtime.
Everything before and after it runs locally.

**Before running.** Set the runtime to GPU, then upload

```
tokens_schaefer100.npy     from results_mamba/
session_feats.npy          from results_mamba/
```

and clone MEG-Mamba for its weights. Output is `mamba_features.npy`, which you download
and put back in `results_mamba/` for notebook 03.

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv,noheader

# mamba_ssm on PyPI does NOT contain Mamba3. MEG-Mamba pins it to commit 316ed60.
# This mirrors their envs/install_mamba3.sh. Expect 15 to 25 minutes, it compiles
# CUDA kernels from source.
#
# Their arch list is "8.0 8.9" (A100, L4). 7.5 is added here for T4, but Mamba kernels
# generally want Ampere or newer, so a T4 may fail at import or at runtime even if the
# build succeeds. If it does, switch to an L4 or A100 runtime.
import os
os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5 8.0 8.9'
os.environ['MAMBA_FORCE_BUILD'] = 'TRUE'
os.environ['CAUSAL_CONV1D_FORCE_BUILD'] = 'TRUE'

!pip install -q ninja setuptools packaging numpy
!pip install -q --no-build-isolation --no-cache-dir "git+https://github.com/state-spaces/mamba.git@316ed60"
!pip install -q --no-build-isolation --no-deps --no-cache-dir causal-conv1d
!git clone -q https://github.com/OHBA-analysis/MEG-Mamba.git

import torch, mamba_ssm
from mamba_ssm import Mamba3
print(f'OK  torch {torch.__version__}  mamba_ssm {mamba_ssm.__version__}')

In [ ]:
import sys, numpy as np, torch
sys.path.insert(0, '/content/MEG-Mamba')
from model import MEGMamba

tokens = np.load('tokens_schaefer100.npy')      # (161, 15000, 100) uint8
feats  = np.load('session_feats.npy')           # (161, 100, 292)
print(f'tokens {tokens.shape}  session features {feats.shape}')

model = MEGMamba.from_pretrained('/content/MEG-Mamba/model', device='cuda')
model.eval()
model.set_session_features(torch.tensor(feats, dtype=torch.float32, device='cuda'))
print('model loaded, session features set')

## What gets extracted

`forward` is `embed` then `backbone` then `readout`. The readout projects to next-token
logits, which is the generative objective and not what we want. Stopping after `backbone`
gives the hidden representation, the same choice as taking `blocks[:-1]` from BrainOmni.

Each subject gives 100 channel sequences of 15000 tokens. We window them, mean-pool over
time within each window, then mean over windows, leaving one vector per channel. Flattening
100 channels by 256 dims gives a 25600 dimensional feature per subject.

In [ ]:
SEQ_LEN  = 1000      # tokens per window
MAX_WIN  = 12        # windows per subject, capped for memory
N_PARCEL = tokens.shape[2]

@torch.no_grad()
def extract(i):
    t = torch.tensor(tokens[i].T, dtype=torch.long, device='cuda')   # (100, 15000)
    cid = torch.arange(N_PARCEL, device='cuda')
    sid = torch.full((N_PARCEL,), i, dtype=torch.long, device='cuda')
    starts = list(range(0, t.shape[1] - SEQ_LEN, SEQ_LEN))[:MAX_WIN]
    acc = torch.zeros(N_PARCEL, model.d_model if hasattr(model, 'd_model') else 256,
                      device='cuda')
    for s in starts:
        h = model.embed(t[:, s:s+SEQ_LEN], cid, sid)
        h = model.backbone(h)              # (100, SEQ_LEN, d_model)
        acc += h.mean(dim=1)               # pool over time
    return (acc / len(starts)).flatten().cpu().numpy()

out = []
for i in range(len(tokens)):
    out.append(extract(i))
    if (i + 1) % 20 == 0:
        print(f'  {i+1}/{len(tokens)}', flush=True)

X = np.stack(out).astype(np.float32)
print(f'\nfeatures {X.shape}')
print(f'finite {np.isfinite(X).all()}   range {X.min():.3f} to {X.max():.3f}')
np.save('mamba_features.npy', X)

from google.colab import files
files.download('mamba_features.npy')